# EDA — 상권 쇠퇴 진단 분석

[기획안_설계.md](기획안_설계.md)에서 확정한 KPI 3개(순증감율, 폐업률 증감, 프랜차이즈 침투율)를 실제로 계산하기 위한 노트북.

**진행 방식**: 단계별로 나눠서 하나씩 진행. 이 노트북은 매 단계마다 이어서 셀을 추가하는 방식으로 자란다.

---
## STEP 1. 상권 × 분기 기본 집계

`stores` 데이터는 `상권 × 분기 × 업종(100개)`이 가장 잘게 쪼개진 단위임.
[기획안_설계.md](기획안_설계.md) 3번 규칙에 따라, KPI는 상권 단위로 판정하기로 했으므로 **업종 100개를 먼저 합산**해서 `상권 × 분기` 단위 표를 만드는 게 첫 단계.

이번 단계에서 하는 일:
1. 데이터 로드
2. 상권×분기 단위로 groupby 합산
3. 결과가 예상한 크기(1,650개 상권 × 4개 분기 = 최대 6,600행)인지 검증

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

DATA_DIR = "../data"

# stores와 master는 cp949(euc-kr) 인코딩 — CLAUDE.md 기술 주의사항 참고
df_stores = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_stores_2025.csv", encoding="cp949")
df_master = pd.read_csv(f"{DATA_DIR}/seoul_commercial_district_master_2025.csv", encoding="cp949")

print("stores:", df_stores.shape)
print("master:", df_master.shape)

stores: (304775, 12)
master: (1650, 11)


### 검증 1: 상권 수 기준이 예상과 맞는지 확인

master 파일에 상권이 1,650개 있고, 분기는 4개(20251~20254)이므로 **이론상 최대 1,650 × 4 = 6,600개 조합**이 나와야 함 (실제 집계 후 행 개수는 뒤에서 확인).

단, 모든 상권이 모든 분기에 데이터가 있으리라는 보장은 없음 (예: 신생 상권이라 특정 분기엔 점포가 아예 없어서 stores 파일에 행 자체가 없을 수 있음).

In [2]:
n_상권 = df_master["상권_코드"].nunique()
n_분기 = df_stores["기준_년분기_코드"].nunique()
n_상권_stores = df_stores["상권_코드"].nunique()

print(f"master 기준 전체 상권 수: {n_상권}")
print(f"stores 기준 분기 수: {n_분기}")
print(f"stores 기준 상권 수: {n_상권_stores}")
print(f"이론상 최대 조합 수 (상권 x 분기): {n_상권 * n_분기}")

master 기준 전체 상권 수: 1650
stores 기준 분기 수: 4
stores 기준 상권 수: 1650
이론상 최대 조합 수 (상권 x 분기): 6600


### 검증 2: 분기별로 상권이 몇 개씩 잡히는지 확인

혹시 특정 분기만 유독 상권 수가 적게 나오면, 그 분기 데이터에 문제가 있다는 신호일 수 있음.

In [3]:
df_stores.groupby("기준_년분기_코드")["상권_코드"].nunique()

기준_년분기_코드
20251    1650
20252    1650
20253    1650
20254    1650
Name: 상권_코드, dtype: int64

### 검증 3: 전체_점포_수 = 일반_점포_수 + 프랜차이즈_점포_수 공식이 실제로 맞는지 확인

[데이터_명세.md](데이터_명세.md)에 이 공식이 문서화되어 있는데, 원본(집계 전, 업종 단위) 레벨에서 실제로 맞는지 검산해보는 단계. 집계는 이 검증이 끝난 뒤, 보정까지 마친 컬럼으로 한 번만 수행한다(아래 STEP 1 재작업 참고).

In [4]:
# 원본(집계 전, 업종 단위) 레벨에서 공식이 맞는지 확인
# != : '같지 않다'는 비교 연산자. 결과는 True/False로 이루어진 열
mismatch = df_stores[df_stores["전체_점포_수"] != (df_stores["일반_점포_수"] + df_stores["프랜차이즈_점포_수"])]

print(f"공식이 안 맞는 행 개수: {len(mismatch)} / 전체 {len(df_stores)}행 ({len(mismatch)/len(df_stores)*100:.1f}%)")
mismatch.head()

공식이 안 맞는 행 개수: 68827 / 전체 304775행 (22.6%)


,기준_년분기_코드,상권_구분_코드,상권_구분_코드_명,상권_코드,상권_코드_명,서비스_업종_코드,서비스_업종_코드_명,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수
0,20251,A,골목상권,3110001,이북5도청사,CS100001,한식음식점,12,13,0,8,1
3,20251,A,골목상권,3110001,이북5도청사,CS100009,호프-간이주점,2,3,0,33,1
17,20251,A,골목상권,3110001,이북5도청사,CS300002,편의점,0,2,0,0,2
25,20251,A,골목상권,3110002,독립문역 1번,CS100001,한식음식점,6,7,0,0,1
26,20251,A,골목상권,3110002,독립문역 1번,CS100005,제과점,0,3,0,0,3


### 검증 3 결과 파헤치기: 왜 22.6%나 공식이 깨질까?

공식(전체 = 일반 + 프랜차이즈)이 안 맞는 행이 원본(집계 전) 레벨에서부터 30만 행 중 약 6.9만 행(22.6%) 있었음. 서울시 공식 데이터 설명 페이지에는 이 공식이 맞다고 나와 있어서, 우리 쪽 파싱이나 계산 문제인지 원본 파일 자체 문제인지 직접 원인을 추적함.

**추적 결과 (원본 CSV 바이트를 직접 열어 대조까지 완료, 이후 재검증으로 원인을 한 번 더 정밀화함):**
- 파싱 오류 아님 — `csv` 모듈로 원본 바이트를 직접 읽어도 동일한 값이 나옴
- **진짜 근본 원인**: `일반_점포_수 = 전체_점포_수 + 폐업_점포_수` 가 30만 행 **전부에서 100% 성립**하는 항등식임 (대조군 무작위 셔플 검증까지 완료해 우연이 아님을 확인). 즉 `일반_점포_수`는 "전체 점포 수"가 아니라 **"폐업분을 포함한 값(폐업 반영 전 스냅샷으로 추정)"**이라는 서로 다른 정의를 가진 컬럼이었음. 서울시 공식 설명("전체=일반+프랜차이즈")은 프랜차이즈=0이고 폐업=0인 행(23만 건)에서만 우연히 성립함
- 이와는 별개로 **`프랜차이즈_점포_수` 컬럼 자체도 오류가 있음**: `프랜차이즈 > 전체_점포_수`인 행이 4.8%(14,705건)나 있어 절대 개수일 수 없음이 확인됨 → 실제 개수 대신 **'프랜차이즈 비율(%)'이 잘못 들어간 원본 데이터 오류**
- 증거: `실제_프랜차이즈_개수 = round(일반_점포_수 × 프랜차이즈_점포_수 / 100)` 로 역산한 뒤, 그 값으로 다시 비율을 계산해 원래 컬럼값과 비교하는 '라운드트립 검증'을 했더니 **95.6%가 정확히 일치**함 (대조군으로 프랜차이즈_점포_수 값을 무작위로 섞어서 같은 계산을 했을 때는 29.2%만 일치 — 실제 데이터가 우연보다 압도적으로 높은 일치율을 보여 오류 가설이 맞다는 근거)
- 보정 모수로 `일반_점포_수`(94.3%대)가 `전체_점포_수`보다 정확함을, 두 값이 실제로 갈라지는 구간(폐업>0인 행)만 따로 떼어 재확인함: 일반 기준 62.3% vs 전체 기준 14.9%

**추가로 발견된 동일 유형 오류**: 같은 방식으로 확인해보니 `개업_점포_수`도 5.3%(16,113행)가 `전체_점포_수`보다 큰 이상값을 가지며, 같은 라운드트립 검증에서 94.3% 일치 — **개업_점포_수에도 같은 '비율(%) 혼입' 오류**가 있음이 확인됨. 반면 `폐업_점포_수`는 라운드트립이 1.6%만 일치해 이 오류 유형이 아니며, 폐업_점포_수 자체는 정상 범위(1~84개, 대부분 1~4개)였음 — `전체_점포_수` 쪽이 실제보다 작게 찍혀서 상대적으로 커 보인 것으로 확인됨.

**결정된 처리 방침**:
1. **프랜차이즈_점포_수**: `round(일반_점포_수 × 프랜차이즈_점포_수 / 100)`로 보정
2. **전체_점포_수**: `max(원본 전체_점포_수, 일반_점포_수_보정 + 프랜차이즈_점포_수_보정)`으로 보정 — 원본값이 실제보다 작게 찍힌 경우를 함께 해결
3. **개업_점포_수**: 별도 보정 공식은 적용하지 않고(불확실성이 남아 보수적으로 접근), 보정된 전체_점포_수를 초과하는 값만 전체_점포_수로 클리핑

In [5]:
# 원본(집계 전, 업종 단위) df_stores에 보정 컬럼을 추가한다.
# 왜 집계 후가 아니라 집계 전에 보정하냐면: 오류가 섞인 값을 먼저 합산해버리면
# 나중에 되돌릴 수 없기 때문. 반드시 가장 잘게 쪼개진 단위에서 고치고 나서 합산해야 함.

df_stores_fixed = df_stores.copy()
import numpy as np

# --- 1) 프랜차이즈_점포_수 보정 ---
# .round(0) : 소수점을 반올림. .astype(int) : 정수형으로 변환 (점포 수는 정수여야 하므로)
전체_보정후_예상값 = (df_stores_fixed["일반_점포_수"] * df_stores_fixed["프랜차이즈_점포_수"] / 100).round(0).astype(int)

# 공식이 원래 맞는 행(전체 == 일반+프랜차이즈)은 원본 프랜차이즈_점포_수를 그대로 쓰고,
# 공식이 깨진 행만 보정값으로 교체한다.
# pandas의 np.where(조건, 참일때 값, 거짓일때 값) 함수를 사용
원래_공식_성립 = df_stores_fixed["전체_점포_수"] == (df_stores_fixed["일반_점포_수"] + df_stores_fixed["프랜차이즈_점포_수"])

df_stores_fixed["프랜차이즈_점포_수_보정"] = np.where(
    원래_공식_성립,
    df_stores_fixed["프랜차이즈_점포_수"],       # 공식이 이미 맞으면 원본값 그대로
    전체_보정후_예상값                              # 공식이 깨졌으면 보정값 사용
)

# 일반_점포_수도 다시 계산: 전체 - 프랜차이즈(보정) = 일반(보정)
df_stores_fixed["일반_점포_수_보정"] = df_stores_fixed["전체_점포_수"] - df_stores_fixed["프랜차이즈_점포_수_보정"]

# 극소수 케이스(전체_점포_수 자체가 0/1처럼 원래 신뢰할 수 없는 값이라
# 보정을 거쳐도 일반_점포_수_보정이 음수로 나오는 행)는 0으로 클리핑한다.
# .clip(lower=0) : 값이 0보다 작으면 0으로, 0 이상이면 그대로 둔다
음수_개수_보정전 = (df_stores_fixed["일반_점포_수_보정"] < 0).sum()
df_stores_fixed["일반_점포_수_보정"] = df_stores_fixed["일반_점포_수_보정"].clip(lower=0)

# --- 2) 전체_점포_수 보정 ---
# 일반_점포_수_보정이 사실상 진짜 전체값에 가깝다는 걸 확인했으므로,
# 원본 전체_점포_수가 (일반_보정 + 프랜차이즈_보정)보다 작게 찍힌 경우 더 큰 값으로 대체한다.
# np.maximum(a, b) : 두 값 중 더 큰 값을 선택 (원소별로)
전체_후보 = df_stores_fixed["일반_점포_수_보정"] + df_stores_fixed["프랜차이즈_점포_수_보정"]
전체_점포_수_늘어난행 = (전체_후보 > df_stores_fixed["전체_점포_수"]).sum()
df_stores_fixed["전체_점포_수_보정"] = np.maximum(df_stores_fixed["전체_점포_수"], 전체_후보)

# --- 3) 개업_점포_수 클리핑 ---
# 개업_점포_수도 프랜차이즈처럼 비율(%) 혼입 오류가 있음을 통계적으로 확인했으나
# (라운드트립 94.3% 일치, 대조군 32.2%로 프랜차이즈와 동일 수준의 근거 확보),
# 실제로 공식 보정을 적용해보면 이상행(5.3%)이 원본 개업 총량의 88.7%를 차지하고 있어
# 위험 상권 판정이 299개->439개로 급변하는 것을 확인함.
# 통계적 근거는 있으나 결과 영향이 지나치게 커서, 예선 단계에서는 보수적으로
# 공식 보정을 적용하지 않고 클리핑(전체_점포_수 초과분만 눌러줌)만 유지하기로 결정.
개업_클리핑_대상 = (df_stores_fixed["개업_점포_수"] > df_stores_fixed["전체_점포_수_보정"]).sum()
df_stores_fixed["개업_점포_수_보정"] = np.minimum(df_stores_fixed["개업_점포_수"], df_stores_fixed["전체_점포_수_보정"])

print("[프랜차이즈] 원래 공식이 맞았던 행:", 원래_공식_성립.sum(), "/", len(df_stores_fixed))
print("[프랜차이즈] 클리핑으로 0 처리한 음수 행:", 음수_개수_보정전, f"({음수_개수_보정전/len(df_stores_fixed)*100:.3f}%)")
print("[전체_점포_수] 더 큰 값으로 늘어난 행:", 전체_점포_수_늘어난행, f"({전체_점포_수_늘어난행/len(df_stores_fixed)*100:.2f}%)")
print("[개업_점포_수] 전체_점포_수_보정으로 클리핑된 행:", 개업_클리핑_대상, f"({개업_클리핑_대상/len(df_stores_fixed)*100:.2f}%)")

[프랜차이즈] 원래 공식이 맞았던 행: 235948 / 304775
[프랜차이즈] 클리핑으로 0 처리한 음수 행: 388 (0.127%)
[전체_점포_수] 더 큰 값으로 늘어난 행: 388 (0.13%)
[개업_점포_수] 전체_점포_수_보정으로 클리핑된 행: 16113 (5.29%)


In [6]:
# 보정 검증: 전체(보정) = 일반(보정) + 프랜차이즈(보정) 공식이 이제 얼마나 맞는지 확인
mismatch_after_fix = df_stores_fixed[
    df_stores_fixed["전체_점포_수_보정"] != (df_stores_fixed["일반_점포_수_보정"] + df_stores_fixed["프랜차이즈_점포_수_보정"])
]
print(f"보정 후에도 공식이 안 맞는 행: {len(mismatch_after_fix)} / {len(df_stores_fixed)} ({len(mismatch_after_fix)/len(df_stores_fixed)*100:.3f}%)")

# 음수가 완전히 사라졌는지 최종 확인
negative_general = df_stores_fixed[df_stores_fixed["일반_점포_수_보정"] < 0]
print(f"일반_점포_수_보정이 음수인 행 (0이어야 함): {len(negative_general)}")

# 개업_점포_수_보정이 전체_점포_수_보정을 넘는 행이 없는지 확인 (클리핑이 잘 됐는지)
개업_초과 = df_stores_fixed[df_stores_fixed["개업_점포_수_보정"] > df_stores_fixed["전체_점포_수_보정"]]
print(f"개업_점포_수_보정 > 전체_점포_수_보정 인 행 (0이어야 함): {len(개업_초과)}")

보정 후에도 공식이 안 맞는 행: 0 / 304775 (0.000%)
일반_점포_수_보정이 음수인 행 (0이어야 함): 0
개업_점포_수_보정 > 전체_점포_수_보정 인 행 (0이어야 함): 0


### 상권×분기 집계 만들기 (보정된 컬럼 기준, 한 번만 수행)

위에서 업종 단위로 보정을 마친 `df_stores_fixed`를 상권×분기 단위로 groupby 합산한다. **앞으로 KPI 계산은 이 `df_agg`를 기준으로 진행한다.**

`groupby(기준열)` : 지정한 열의 값이 같은 행들을 하나의 그룹으로 묶는다.
예를 들어 `groupby(["상권_코드", "기준_년분기_코드"])`를 하면, 같은 상권·같은 분기에 속한 100개 업종 행들이 한 그룹이 됨.
그 다음 `[열들].sum()`으로 그룹별 합계를 구하면, `(상권_코드, 기준_년분기_코드)` 조합 하나당 한 행이 되고 그 안의 100개 업종 점포 수가 다 더해진 값이 나온다.

In [7]:
# groupby 기준이 될 열: 상권_코드, 기준_년분기_코드 (이 둘이 같으면 같은 그룹)
# 상권_구분_코드_명도 같이 남겨두면 나중에 '골목상권 vs 발달상권' 비교할 때 편함
# → as_index=False : groupby 기준 열들을 인덱스가 아니라 일반 컬럼으로 유지 (다루기 편하게)
group_keys = ["상권_코드", "상권_코드_명", "상권_구분_코드_명", "기준_년분기_코드"]
sum_cols_fixed = ["전체_점포_수_보정", "일반_점포_수_보정", "프랜차이즈_점포_수_보정", "개업_점포_수_보정", "폐업_점포_수"]

df_agg = df_stores_fixed.groupby(group_keys, as_index=False)[sum_cols_fixed].sum()

# 이후 KPI 계산 코드에서 다루기 쉽도록, 보정된 컬럼명을 기존 이름으로 정리
df_agg = df_agg.rename(columns={
    "전체_점포_수_보정": "전체_점포_수",
    "일반_점포_수_보정": "일반_점포_수",
    "프랜차이즈_점포_수_보정": "프랜차이즈_점포_수",
    "개업_점포_수_보정": "개업_점포_수",
})

print("집계 완료:", df_agg.shape)

# 검증 1: 예상한 크기(1,650개 상권 × 4개 분기 = 최대 6,600행)인지 확인
print(f"이론상 최대 행 수 (상권 x 분기): {n_상권 * n_분기}, 실제: {len(df_agg)}")

# 검증 2: 상권×분기 단위에서도 공식이 잘 맞는지 확인
mismatch_final = df_agg[df_agg["전체_점포_수"] != (df_agg["일반_점포_수"] + df_agg["프랜차이즈_점포_수"])]
print(f"상권×분기 집계 기준, 공식이 안 맞는 행: {len(mismatch_final)} / {len(df_agg)}")

# 검증 3: 개업_점포_수가 전체_점포_수를 넘는 행이 없는지 확인
개업_초과_후 = df_agg[df_agg["개업_점포_수"] > df_agg["전체_점포_수"]]
print(f"집계 후 개업 > 전체 인 행: {len(개업_초과_후)}")

df_agg.head(10)

집계 완료: (6600, 9)
이론상 최대 행 수 (상권 x 분기): 6600, 실제: 6600
상권×분기 집계 기준, 공식이 안 맞는 행: 0 / 6600
집계 후 개업 > 전체 인 행: 0


,상권_코드,상권_코드_명,상권_구분_코드_명,기준_년분기_코드,전체_점포_수,일반_점포_수,프랜차이즈_점포_수,개업_점포_수,폐업_점포_수
0,3001491,이태원 관광특구,관광특구,20251,1689,1662,27,108,96
1,3001491,이태원 관광특구,관광특구,20252,1680,1636,44,112,93
2,3001491,이태원 관광특구,관광특구,20253,1677,1635,42,65,93
3,3001491,이태원 관광특구,관광특구,20254,1680,1641,39,60,93
4,3001492,명동 남대문 북창동 다동 무교동 관광특구,관광특구,20251,8495,8375,120,112,548
5,3001492,명동 남대문 북창동 다동 무교동 관광특구,관광특구,20252,8455,8344,111,108,558
6,3001492,명동 남대문 북창동 다동 무교동 관광특구,관광특구,20253,8460,8297,163,138,554
7,3001492,명동 남대문 북창동 다동 무교동 관광특구,관광특구,20254,8453,8305,148,106,544
8,3001493,동대문패션타운 관광특구,관광특구,20251,13310,13283,27,119,237
9,3001493,동대문패션타운 관광특구,관광특구,20252,13069,13034,35,100,238


---
## STEP 2. KPI 3개 계산

[기획안_설계.md](기획안_설계.md) 3번 항목에서 확정한 KPI를 `df_agg`(보정된 상권×분기 집계)에 실제로 계산해서 붙인다.

| ID | 지표명 | 산식 |
|---|---|---|
| KPI-1 | 순증감율 | (개업_점포_수 − 폐업_점포_수) / 전체_점포_수 |
| KPI-2 | 폐업률 증감 | 이번 분기 폐업률 − 전분기 폐업률 (폐업률 = 폐업_점포_수 / 전체_점포_수) |
| KPI-3 | 프랜차이즈 침투율 | 프랜차이즈_점포_수(보정) / 전체_점포_수 |

KPI-1, KPI-3은 한 분기 안에서 바로 계산 가능하지만, KPI-2는 "전분기 값"이 필요하므로 같은 상권 안에서 분기 순서대로 정렬한 뒤 비교해야 함.

### 계산 전 확인: 전체_점포_수가 0인 행이 있는가?

순증감율·프랜차이즈 침투율 모두 분모가 `전체_점포_수`임. 분모가 0이면 "0으로 나누기" 에러가 나거나 무한대(inf)/결측(NaN)이 나오므로, 계산 전에 몇 개나 있는지 먼저 확인.

In [8]:
zero_total = df_agg[df_agg["전체_점포_수"] == 0]
print(f"전체_점포_수가 0인 상권×분기 행: {len(zero_total)} / {len(df_agg)}")
zero_total[["상권_코드_명", "상권_구분_코드_명", "기준_년분기_코드", "전체_점포_수", "개업_점포_수", "폐업_점포_수"]]

전체_점포_수가 0인 상권×분기 행: 0 / 6600


,상권_코드_명,상권_구분_코드_명,기준_년분기_코드,전체_점포_수,개업_점포_수,폐업_점포_수


### KPI-1(순증감율), KPI-3(프랜차이즈 침투율) 계산

분모가 0인 행이 있다면, 그 행의 KPI 값은 "계산 불가"라는 의미로 결측치(NaN)로 남겨두는 게 맞음 (0으로 억지로 채우면 "위축률 0%=변화없음"처럼 잘못 해석될 수 있어서 더 위험함).

`.replace(0, np.nan)`으로 분모의 0을 NaN으로 바꿔두면, 나눗셈 결과도 자동으로 NaN이 됨 (0/0 대신 명시적으로 "모른다"를 표현).

In [9]:
# 분모로 쓸 전체_점포_수의 0을 NaN으로 바꾼 버전을 따로 만든다 (원본 전체_점포_수 컬럼은 그대로 둠)
분모_안전 = df_agg["전체_점포_수"].replace(0, np.nan)

# KPI-1: 순증감율 = (개업 - 폐업) / 전체
df_agg["KPI1_순증감율"] = (df_agg["개업_점포_수"] - df_agg["폐업_점포_수"]) / 분모_안전

# KPI-3: 프랜차이즈 침투율 = 프랜차이즈(보정) / 전체
df_agg["KPI3_프랜차이즈침투율"] = df_agg["프랜차이즈_점포_수"] / 분모_안전

# .describe() 로 값 범위가 상식적인지 확인 (순증감율은 보통 -1~1 사이, 침투율은 0~1 사이여야 함)
df_agg[["KPI1_순증감율", "KPI3_프랜차이즈침투율"]].describe()

,KPI1_순증감율,KPI3_프랜차이즈침투율
count,6600.000000,6600.000000
mean,-0.001433,0.021709
std,0.085067,0.019812
min,-0.481481,0.000000
25%,-0.050000,0.009513
50%,0.000000,0.019741
75%,0.045455,0.030160
max,0.583333,0.479167


### KPI-2(폐업률 증감) 계산 — 전분기 값이 필요한 지표

이 지표는 "이번 분기 폐업률 - 전분기 폐업률"이라서, 같은 상권 안에서 분기 순서대로 줄을 세운 뒤 바로 위 행(전분기)의 값을 가져와야 함.

- `.sort_values(["상권_코드", "기준_년분기_코드"])` : 상권별로, 그 안에서 분기 오름차순으로 정렬
- `.groupby("상권_코드")["폐업률"].shift(1)` : 그룹(상권) 안에서 한 칸 아래로 밀어서 '바로 이전 분기 값'을 지금 행으로 가져옴 (첫 분기는 이전이 없으므로 NaN)

In [10]:
# 먼저 이번 분기 폐업률 자체를 계산 (분모는 위에서 만든 0->NaN 처리된 값 재사용)
df_agg["폐업률"] = df_agg["폐업_점포_수"] / 분모_안전

# 상권별 + 분기 오름차순으로 정렬 (shift는 행 순서에 의존하므로 정렬이 반드시 먼저 필요)
df_agg = df_agg.sort_values(["상권_코드", "기준_년분기_코드"]).reset_index(drop=True)

# 그룹(상권)별로 바로 이전 분기의 폐업률을 가져옴
df_agg["폐업률_전분기"] = df_agg.groupby("상권_코드")["폐업률"].shift(1)

# KPI-2: 이번 분기 폐업률 - 전분기 폐업률
df_agg["KPI2_폐업률증감"] = df_agg["폐업률"] - df_agg["폐업률_전분기"]

print("1분기(20251)는 전분기가 없으므로 KPI2가 결측이어야 함:")
print(df_agg.groupby("기준_년분기_코드")["KPI2_폐업률증감"].apply(lambda x: x.isna().sum()))
print()
df_agg[["상권_코드_명", "기준_년분기_코드", "폐업률", "폐업률_전분기", "KPI2_폐업률증감"]].head(8)

1분기(20251)는 전분기가 없으므로 KPI2가 결측이어야 함:
기준_년분기_코드
20251    1650
20252       0
20253       0
20254       0
Name: KPI2_폐업률증감, dtype: int64



,상권_코드_명,기준_년분기_코드,폐업률,폐업률_전분기,KPI2_폐업률증감
0,이태원 관광특구,20251,0.056838,NaN,NaN
1,이태원 관광특구,20252,0.055357,0.056838,-0.001481
2,이태원 관광특구,20253,0.055456,0.055357,0.000099
3,이태원 관광특구,20254,0.055357,0.055456,-0.000099
4,명동 남대문 북창동 다동 무교동 관광특구,20251,0.064509,NaN,NaN
5,명동 남대문 북창동 다동 무교동 관광특구,20252,0.065996,0.064509,0.001488
6,명동 남대문 북창동 다동 무교동 관광특구,20253,0.065485,0.065996,-0.000512
7,명동 남대문 북창동 다동 무교동 관광특구,20254,0.064356,0.065485,-0.001129


### KPI 3개 최종 확인 + 위험 상권 판정 미리보기

[기획안_설계.md](기획안_설계.md) 4번 항목에서 정한 위험 판정 기준을 그대로 적용해본다:
- 기준 분기: 최신 분기(20254)
- 조건: `KPI1_순증감율 < 0` **AND** `KPI2_폐업률증감 > 0`

아직 최소 표본 기준(이상치 처리)은 적용하지 않은 1차 결과이므로, 정확한 위험 상권 "확정" 리스트가 아니라 감(느낌)을 잡는 용도.

In [11]:
# 최신 분기(20254)만 따로 추출
df_최신분기 = df_agg[df_agg["기준_년분기_코드"] == 20254].copy()

# 위험 판정 조건: 순증감율 < 0 이면서 동시에 폐업률증감 > 0
위험_조건 = (df_최신분기["KPI1_순증감율"] < 0) & (df_최신분기["KPI2_폐업률증감"] > 0)
df_위험상권 = df_최신분기[위험_조건]

print(f"최신 분기(20254) 전체 상권 수: {len(df_최신분기)}")
print(f"위험 판정 기준 통과 상권 수: {len(df_위험상권)} ({len(df_위험상권)/len(df_최신분기)*100:.1f}%)")
print()

# KPI2(악화 속도) 기준 내림차순 정렬해서 가장 위험한 상위 10개 미리보기
df_위험상권.sort_values("KPI2_폐업률증감", ascending=False)[
    ["상권_코드_명", "상권_구분_코드_명", "KPI1_순증감율", "KPI2_폐업률증감", "KPI3_프랜차이즈침투율"]
].head(10)

최신 분기(20254) 전체 상권 수: 1650
위험 판정 기준 통과 상권 수: 299 (18.1%)



,상권_코드_명,상권_구분_코드_명,KPI1_순증감율,KPI2_폐업률증감,KPI3_프랜차이즈침투율
907,의릉,골목상권,-0.139535,0.050740,0.046512
1567,연산군묘,골목상권,-0.272727,0.041958,0.000000
1767,상계제일중학교,골목상권,-0.184615,0.041758,0.076923
6223,고척골목시장상점가(그라운드 고척),전통시장,-0.019802,0.038515,0.069307
4295,강동성심병원,골목상권,-0.481481,0.037037,0.037037
3195,대림썬프라자,골목상권,-0.030303,0.034759,0.030303
6139,신정3동 골목시장,전통시장,-0.175000,0.032143,0.000000
3295,신대방역 3번,골목상권,-0.052632,0.030263,0.052632
3535,미림여고,골목상권,-0.048780,0.027294,0.024390
5843,삼양골목시장(삼양시장),전통시장,-0.041667,0.027186,0.058333


---
## STEP 3. 자치구 단위 집계 (화면 1용 재료 준비)

[기획안_설계.md](기획안_설계.md) 2번 항목에서 확정한 대로, 우리 분석은 `자치구 → 상권` 두 계층으로 구성됨.
STEP 1·2에서는 상권 단위(화면2용)까지만 만들었으니, 이번 STEP에서는 화면1(자치구 단위 요약)에 쓸 데이터를 준비한다.

하는 일:
1. `master` 파일과 조인해서 각 상권이 어느 자치구에 속하는지 붙이기
2. 자치구×분기 단위로 점포 수를 다시 합산 (상권 단위 KPI를 평균 내는 게 아니라, [기획안_설계.md](기획안_설계.md) 3번과 같은 원칙으로 점포 수 자체를 다시 합산한 뒤 비율 계산)
3. 자치구 단위 KPI 3개 계산 + 자치구별 위험 상권 개수 집계

### master와 조인해서 자치구 정보 붙이기

`pd.merge(왼쪽표, 오른쪽표, on="공통열", how="left")` : 두 표에서 `상권_코드` 값이 같은 행끼리 짝지어 합친다.
`how="left"`는 왼쪽 표(df_agg) 기준으로 합치고, 오른쪽에 짝이 없는 행이 있어도 왼쪽 행은 그대로 남긴다는 뜻.

In [12]:
# df_agg에서 자치구 집계에 필요한 '순수 점포 수' 컬럼만 골라서 조인 (KPI 계산 중간값은 제외)
점포수_컬럼 = ["상권_코드", "기준_년분기_코드", "전체_점포_수", "개업_점포_수", "폐업_점포_수", "프랜차이즈_점포_수"]

df_구별 = pd.merge(
    df_agg[점포수_컬럼],
    df_master[["상권_코드", "자치구_코드_명"]],
    on="상권_코드",
    how="left",
)

# 조인 후 자치구_코드_명이 비어있는(=master에 없는 상권_코드) 행이 있는지 확인
print("자치구_코드_명이 결측인 행:", df_구별["자치구_코드_명"].isna().sum(), "/", len(df_구별))
df_구별.head()

자치구_코드_명이 결측인 행: 0 / 6600


,상권_코드,기준_년분기_코드,전체_점포_수,개업_점포_수,폐업_점포_수,프랜차이즈_점포_수,자치구_코드_명
0,3001491,20251,1689,108,96,27,용산구
1,3001491,20252,1680,112,93,44,용산구
2,3001491,20253,1677,65,93,42,용산구
3,3001491,20254,1680,60,93,39,용산구
4,3001492,20251,8495,112,548,120,중구


### 자치구 × 분기 단위로 점포 수 재집계

상권 단위 KPI(비율)를 평균 내지 않고, [기획안_설계.md](기획안_설계.md) 3번과 같은 원칙(먼저 점포 수를 합산한 뒤 비율 계산)을 자치구 레벨에서도 그대로 적용한다.
25개 자치구 × 4개 분기 = 최대 100행이 나와야 함.

In [13]:
구별_sum_cols = ["전체_점포_수", "개업_점포_수", "폐업_점포_수", "프랜차이즈_점포_수"]

df_구별_agg = df_구별.groupby(["자치구_코드_명", "기준_년분기_코드"], as_index=False)[구별_sum_cols].sum()

n_자치구 = df_master["자치구_코드_명"].nunique()
print(f"자치구 수: {n_자치구}, 분기 수: {n_분기}, 이론상 최대 행 수: {n_자치구 * n_분기}")
print(f"실제 집계된 행 수: {len(df_구별_agg)}")

df_구별_agg.head(10)

자치구 수: 25, 분기 수: 4, 이론상 최대 행 수: 100
실제 집계된 행 수: 100


,자치구_코드_명,기준_년분기_코드,전체_점포_수,개업_점포_수,폐업_점포_수,프랜차이즈_점포_수
0,강남구,20251,50593,4263,3862,914
1,강남구,20252,50413,3341,3837,1091
2,강남구,20253,50157,3496,3815,1100
3,강남구,20254,50083,3001,3774,978
4,강동구,20251,16113,1800,1711,362
5,강동구,20252,16016,1633,1713,421
6,강동구,20253,15983,1581,1700,427
7,강동구,20254,15885,1583,1675,345
8,강북구,20251,8909,951,869,195
9,강북구,20252,8814,979,865,239


### 자치구 단위 KPI 3개 계산

STEP 2에서 상권 단위로 했던 계산(KPI-1, KPI-3은 바로 계산, KPI-2는 전분기 대비)을 자치구 단위에서 그대로 반복한다.

In [14]:
# 자치구 단위는 상권을 100개씩 합친 값이라 분모가 0이 될 가능성이 거의 없지만,
# 상권 단위 때와 동일한 안전장치를 그대로 적용 (0을 NaN으로)
구별_분모_안전 = df_구별_agg["전체_점포_수"].replace(0, np.nan)

df_구별_agg["KPI1_순증감율"] = (df_구별_agg["개업_점포_수"] - df_구별_agg["폐업_점포_수"]) / 구별_분모_안전
df_구별_agg["KPI3_프랜차이즈침투율"] = df_구별_agg["프랜차이즈_점포_수"] / 구별_분모_안전

# KPI-2: 전분기 대비 폐업률 증감 (자치구별로 분기 순서대로 정렬 후 shift)
df_구별_agg["폐업률"] = df_구별_agg["폐업_점포_수"] / 구별_분모_안전
df_구별_agg = df_구별_agg.sort_values(["자치구_코드_명", "기준_년분기_코드"]).reset_index(drop=True)
df_구별_agg["폐업률_전분기"] = df_구별_agg.groupby("자치구_코드_명")["폐업률"].shift(1)
df_구별_agg["KPI2_폐업률증감"] = df_구별_agg["폐업률"] - df_구별_agg["폐업률_전분기"]

# 값 범위가 상식적인지 확인 (상권 단위 때와 비슷한 범위여야 함)
df_구별_agg[["KPI1_순증감율", "KPI2_폐업률증감", "KPI3_프랜차이즈침투율"]].describe()

,KPI1_순증감율,KPI2_폐업률증감,KPI3_프랜차이즈침투율
count,100.000000,75.000000,100.000000
mean,-0.005536,-0.000593,0.022268
std,0.015361,0.000752,0.004596
min,-0.046525,-0.002131,0.009046
25%,-0.015656,-0.001018,0.019675
50%,-0.005644,-0.000593,0.022449
75%,0.004320,-0.000020,0.025403
max,0.035597,0.000894,0.032577


### 자치구별 위험 상권 개수 (화면1 Section 2 랭킹용 재료)

STEP 2에서 만든 상권 단위 위험 판정 결과(`df_위험상권`, 최신 분기 기준)에 자치구 정보를 붙여서, 자치구마다 위험 상권이 몇 개씩 있는지 센다.
이 값이 화면1의 "자치구별 실적" 막대그래프/지도에 쓰일 핵심 수치.

In [15]:
# 위험 상권(df_위험상권)에 자치구 정보를 붙인다
df_위험상권_구별 = pd.merge(
    df_위험상권[["상권_코드", "상권_코드_명"]],
    df_master[["상권_코드", "자치구_코드_명"]],
    on="상권_코드",
    how="left",
)

# 자치구별 위험 상권 개수를 세고, 전체 상권 대비 비율도 함께 계산
자치구별_위험개수 = df_위험상권_구별.groupby("자치구_코드_명").size().rename("위험_상권_수")
자치구별_전체상권수 = df_master.groupby("자치구_코드_명").size().rename("전체_상권_수")

df_자치구_위험도 = pd.concat([자치구별_전체상권수, 자치구별_위험개수], axis=1).fillna(0)
df_자치구_위험도["위험_상권_수"] = df_자치구_위험도["위험_상권_수"].astype(int)
df_자치구_위험도["위험_비율"] = df_자치구_위험도["위험_상권_수"] / df_자치구_위험도["전체_상권_수"]

# 위험 비율이 높은 순으로 정렬해서 상위 10개 자치구 확인
df_자치구_위험도.sort_values("위험_비율", ascending=False).head(10)

,전체_상권_수,위험_상권_수,위험_비율
자치구_코드_명,,,
동작구,57,17,0.298246
강동구,60,16,0.266667
도봉구,43,10,0.232558
양천구,56,13,0.232143
구로구,60,13,0.216667
송파구,70,15,0.214286
서대문구,62,13,0.209677
은평구,68,14,0.205882
강서구,70,14,0.200000


---
## STEP 4. 문제 지역·패턴 도출 ([기획안_설계.md](기획안_설계.md) 4장 재료)

STEP 3에서 만든 `df_자치구_위험도`, `df_agg`, `df_위험상권`을 그대로 재사용해서 두 가지를 더 본다.

1. 자치구별 위험 비율 **하위** 순위 (STEP 3에서는 상위 10개만 봤음 — 건강한 자치구 확인용)
2. 상권구분(골목상권/발달상권/전통시장/관광특구)별 실적 비교 + 위험 상권 분포

### 4-1. 자치구별 위험 비율 하위 순위 (건강한 자치구)

`df_자치구_위험도`는 이미 STEP 3에서 다 만들어져 있으므로, 정렬 방향만 오름차순(`ascending=True`)으로 바꿔서 하위 10개를 본다.

In [16]:
# 위험 비율이 낮은 순(오름차순)으로 정렬해서 하위 10개(=상대적으로 건강한 자치구) 확인
df_자치구_위험도.sort_values("위험_비율", ascending=True).head(10)

,전체_상권_수,위험_상권_수,위험_비율
자치구_코드_명,,,
금천구,45,5,0.111111
중구,68,8,0.117647
용산구,57,7,0.122807
마포구,77,11,0.142857
동대문구,77,11,0.142857
광진구,62,9,0.145161
관악구,74,11,0.148649
성북구,76,12,0.157895
종로구,74,12,0.162162


### 4-2. 상권구분별 실적 비교

`df_agg`에는 이미 `상권_구분_코드_명`(골목상권/발달상권/전통시장/관광특구)이 있으므로, 최신 분기(20254)만 골라서 이 기준으로 groupby한다.

[기획안_설계.md](기획안_설계.md) 3번 집계 규칙과 동일하게, 먼저 점포 수를 합산한 뒤 KPI-1·KPI-3을 계산한다. KPI-2(폐업률 증감)는 상권 단위에서 이미 계산되어 있으므로 평균으로 요약한다(자치구 집계 때와 동일 패턴).

In [17]:
# 최신 분기(20254)의 상권 단위 데이터만 사용 (df_최신분기는 STEP 2에서 이미 만들어 둔 변수)
구분별_sum_cols = ["전체_점포_수", "개업_점포_수", "폐업_점포_수", "프랜차이즈_점포_수"]

df_구분별_agg = df_최신분기.groupby("상권_구분_코드_명", as_index=False)[구분별_sum_cols].sum()

# 점포 수를 합산한 뒤 KPI-1(순증감율), KPI-3(프랜차이즈 침투율) 재계산
구분별_분모_안전 = df_구분별_agg["전체_점포_수"].replace(0, np.nan)
df_구분별_agg["KPI1_순증감율"] = (df_구분별_agg["개업_점포_수"] - df_구분별_agg["폐업_점포_수"]) / 구분별_분모_안전
df_구분별_agg["KPI3_프랜차이즈침투율"] = df_구분별_agg["프랜차이즈_점포_수"] / 구분별_분모_안전

# KPI-2(폐업률 증감)는 상권 단위에서 이미 계산된 값(df_최신분기["KPI2_폐업률증감"])의 평균으로 요약
구분별_KPI2_평균 = df_최신분기.groupby("상권_구분_코드_명")["KPI2_폐업률증감"].mean()
df_구분별_agg = df_구분별_agg.merge(구분별_KPI2_평균.rename("KPI2_폐업률증감_평균"), on="상권_구분_코드_명")

df_구분별_agg[["상권_구분_코드_명", "전체_점포_수", "KPI1_순증감율", "KPI2_폐업률증감_평균", "KPI3_프랜차이즈침투율"]]

,상권_구분_코드_명,전체_점포_수,KPI1_순증감율,KPI2_폐업률증감_평균,KPI3_프랜차이즈침투율
0,골목상권,159725,0.007870,-0.001196,0.022633
1,관광특구,31014,-0.032695,-0.001029,0.011608
2,발달상권,227547,-0.033778,-0.001180,0.019148
3,전통시장,55677,0.005765,-0.000412,0.017727


### 4-3. 상권구분별 위험 상권 분포

위험 판정된 299개 상권(`df_위험상권`)이 골목상권/발달상권/전통시장/관광특구 중 어디에 몰려있는지 비율로 확인한다. 전체 상권 대비 위험 상권 비율을 구분별로 비교해야 "특정 구분에서 유독 위험 비율이 높다"는 패턴을 잡을 수 있다.

In [18]:
# 상권구분별 전체 상권 수 (master 기준, 분기와 무관하게 상권 하나당 한 번만 세어짐)
구분별_전체상권수 = df_master.groupby("상권_구분_코드_명").size().rename("전체_상권_수")

# 상권구분별 위험 상권 수 (df_위험상권에는 상권_구분_코드_명 컬럼이 이미 있음)
구분별_위험상권수 = df_위험상권.groupby("상권_구분_코드_명").size().rename("위험_상권_수")

df_구분별_위험도 = pd.concat([구분별_전체상권수, 구분별_위험상권수], axis=1).fillna(0)
df_구분별_위험도["위험_상권_수"] = df_구분별_위험도["위험_상권_수"].astype(int)
df_구분별_위험도["위험_비율"] = df_구분별_위험도["위험_상권_수"] / df_구분별_위험도["전체_상권_수"]

# 검증: 위험_상권_수 합계가 STEP 2에서 나온 299개와 일치해야 함
print("상권구분별 위험 상권 수 합계:", df_구분별_위험도["위험_상권_수"].sum(), "(299여야 함)")
print()

df_구분별_위험도.sort_values("위험_비율", ascending=False)

상권구분별 위험 상권 수 합계: 299 (299여야 함)



,전체_상권_수,위험_상권_수,위험_비율
상권_구분_코드_명,,,
발달상권,249,57,0.228916
전통시장,305,53,0.173770
골목상권,1090,188,0.172477
관광특구,6,1,0.166667


---
## STEP 5. 발달상권 위험비율이 왜 제일 높은지 원인 분석

STEP 4에서 예상과 반대로(골목상권/전통시장이 아니라) **발달상권**의 위험 비율이 가장 높게(22.9%) 나온 걸 확인했음.

**가설**: 발달상권은 원래 임대료가 비싸고 트렌드에 민감한 업종(외식·패션 등)이 몰려있어 개폐업 변동폭 자체가 다른 상권구분보다 큼 → 위험 판정 기준(전분기 대비 폐업률 급증)에 더 잘 걸릴 수 있음.

**검증 방법**: `df_agg`(상권×분기, 4개 분기 전체)를 상권구분별로 묶어서 KPI-1(순증감율)·KPI-2(폐업률 증감)의 표준편차를 비교. 표준편차가 크다는 건 분기마다 값이 들쭉날쭉하다는 뜻 = 변동성이 크다는 뜻.

In [19]:
# df_agg는 4개 분기 전체(상권 단위 KPI가 이미 계산되어 있음, STEP 2에서 만든 변수)
# 상권구분별로 묶어서 KPI1_순증감율, KPI2_폐업률증감의 표준편차(std)와 평균(mean)을 같이 확인
df_구분별_변동성 = df_agg.groupby("상권_구분_코드_명")[["KPI1_순증감율", "KPI2_폐업률증감"]].agg(["mean", "std"])

print("=== 상권구분별 KPI1(순증감율)·KPI2(폐업률증감) 평균/표준편차 (4개 분기 전체) ===")
df_구분별_변동성

=== 상권구분별 KPI1(순증감율)·KPI2(폐업률증감) 평균/표준편차 (4개 분기 전체) ===


KPI1_순증감율           KPI2_폐업률증감          
                mean       std       mean       std
상권_구분_코드_명                                         
골목상권        0.000529  0.088713  -0.000495  0.008824
관광특구       -0.039517  0.029799  -0.000347  0.001137
발달상권       -0.022699  0.055655  -0.000756  0.003796
전통시장        0.009667  0.089171  -0.000309  0.010453

### 4개 분기 추이로 다시 확인

표준편차 하나만으로는 "원래 들쭉날쭉한 것"과 "한 방향으로 계속 나빠지는 것"을 구별할 수 없음. 상권구분별로 분기마다 평균 KPI1(순증감율)이 어떻게 움직였는지 직접 찍어봐야 진짜 "지속적 하락 추세"인지 "변동성만 큰 것"인지 판단 가능.

In [20]:
# 상권구분 x 분기별 KPI1_순증감율 평균을 표로 펼쳐서 분기 흐름을 한눈에 비교
# .unstack() : groupby(상권구분, 분기) 결과에서 분기를 열로 펼침 -> 상권구분 하나당 한 행, 분기별 값이 열로 나열됨
df_구분별_분기추이 = df_agg.groupby(["상권_구분_코드_명", "기준_년분기_코드"])["KPI1_순증감율"].mean().unstack()

print("=== 상권구분별 분기별 평균 순증감율(KPI1) 추이 ===")
df_구분별_분기추이

=== 상권구분별 분기별 평균 순증감율(KPI1) 추이 ===


기준_년분기_코드,20251,20252,20253,20254
상권_구분_코드_명,,,,
골목상권,0.014425,-0.001329,-0.004568,-0.006412
관광특구,-0.028718,-0.042618,-0.044185,-0.042547
발달상권,-0.011830,-0.026324,-0.020205,-0.032437
전통시장,0.009267,0.011772,0.013099,0.004530


---
## STEP 6. 가설 3 검증 — 발달상권 내 프랜차이즈 침투율 패턴 (젠트리피케이션 가설)

STEP 5에서 발달상권은 "변동성"이 아니라 **4개 분기 내내 꾸준히 나빠지는 추세**임을 확인했음. 이번엔 그 추세와 프랜차이즈 침투율(KPI-3)이 관련 있는지 본다.

**가설**: 발달상권 중에서도 위험 판정을 받은 상권은, 비위험 발달상권보다 프랜차이즈 침투율이 더 높거나(이미 프랜차이즈로 많이 대체됨) 혹은 최근 분기에 침투율이 빠르게 오르고 있을 것(임대료 상승 → 개인 상점 밀려나고 프랜차이즈 유입 → 상권 획일화·쇠퇴로 이어지는 젠트리피케이션 패턴).

**검증 방법**:
1. 발달상권만 필터링해서 위험/비위험으로 나눔
2. 최신 분기(20254) 기준 두 그룹의 KPI-3(프랜차이즈 침투율) 평균 비교
3. 두 그룹의 4개 분기 프랜차이즈 침투율 추이 비교 (침투율이 오르는 속도 차이 확인)

In [21]:
# 발달상권만 필터링
df_발달상권 = df_agg[df_agg["상권_구분_코드_명"] == "발달상권"].copy()

# 발달상권 중 위험 상권의 상권_코드 집합 (df_위험상권은 STEP 2에서 만든, 최신분기 기준 위험판정 결과)
발달상권_위험코드_집합 = set(df_위험상권[df_위험상권["상권_구분_코드_명"] == "발달상권"]["상권_코드"])
df_발달상권["위험상권_여부"] = df_발달상권["상권_코드"].isin(발달상권_위험코드_집합)

print("발달상권 전체:", df_발달상권["상권_코드"].nunique(), "개")
print("이 중 위험 상권:", len(발달상권_위험코드_집합), "개 (STEP 4에서 나온 57개와 일치해야 함)")
print()

# 최신 분기(20254) 기준, 위험/비위험 그룹별 KPI-3(프랜차이즈 침투율) 평균 비교
df_발달상권_최신 = df_발달상권[df_발달상권["기준_년분기_코드"] == 20254]
print("=== 최신 분기(20254) 발달상권 그룹별 KPI-3(프랜차이즈 침투율) 평균 ===")
print(df_발달상권_최신.groupby("위험상권_여부")["KPI3_프랜차이즈침투율"].mean())

발달상권 전체: 249 개
이 중 위험 상권: 57 개 (STEP 4에서 나온 57개와 일치해야 함)

=== 최신 분기(20254) 발달상권 그룹별 KPI-3(프랜차이즈 침투율) 평균 ===
위험상권_여부
False    0.019903
True     0.018973
Name: KPI3_프랜차이즈침투율, dtype: float64


### 4개 분기 프랜차이즈 침투율 추이 비교

수준(평균값) 차이뿐 아니라, "오르는 속도"도 젠트리피케이션 가설의 핵심이므로 분기별 추이도 함께 본다.

In [22]:
# 위험상권_여부는 최신 분기(20254) 기준으로 판정된 값이지만, 그 이전 분기에도 같은 상권이므로
# "이 상권이 최종적으로 위험 판정을 받았는가"로 4개 분기 전체에 태그를 붙여 추이를 볼 수 있음
df_발달상권_분기추이 = df_발달상권.groupby(["위험상권_여부", "기준_년분기_코드"])["KPI3_프랜차이즈침투율"].mean().unstack(level=0)

print("=== 발달상권 위험/비위험 그룹별 분기별 프랜차이즈 침투율 추이 ===")
df_발달상권_분기추이

=== 발달상권 위험/비위험 그룹별 분기별 프랜차이즈 침투율 추이 ===


위험상권_여부,False,True
기준_년분기_코드,,
20251,0.019056,0.020388
20252,0.023274,0.023813
20253,0.024822,0.024026
20254,0.019903,0.018973


### STEP 5·6 결론

발달상권의 위험 비율이 25개 자치구 상권구분 중 가장 높게(22.9%) 나온 원인을 두 가설로 검증함:

- **가설 1(변동성)** — 기각. 발달상권 KPI 표준편차(0.056)는 골목상권(0.089)·전통시장(0.089)보다 오히려 작음. 대신 4개 분기 내내 순증감율이 -1.2%→-2.6%→-2.0%→-3.2%로 **지속적으로 하락**하는 추세를 확인.
- **가설 3(프랜차이즈 침투)** — 기각. 발달상권 내 위험/비위험 그룹의 프랜차이즈 침투율 수준(1.90% vs 1.99%)과 4개 분기 추이가 거의 동일. 프랜차이즈 유입이 쇠퇴를 이끈다는 근거 없음.

**결론**: 발달상권의 위험은 단발성 변동이나 프랜차이즈 침투가 아니라 **구조적·지속적 쇠퇴 추세**로 보임. 근본 원인(임대료, 유동인구, 소비력 등)은 보유한 3개 CSV(점포수·업종·집객시설)만으로는 규명 불가 — 최종 기획안에서는 "발달상권을 우선 모니터링 대상으로 명시"하는 정책 제언과, 원인 규명에는 외부 데이터(임대료/유동인구)가 필요함을 한계로 서술하기로 함.

---
## STEP 7. 최소 표본 임계값 적용 (이상치 처리)

[기획안_설계.md](기획안_설계.md) 5번에서 미확정으로 남아있던 "표본이 너무 작은 상권" 처리 기준을 결정한다.

**배경**: 위험 판정(순증감율<0 AND 폐업률증감>0)은 비율 계산이라, 점포 수가 극히 적은 상권은 점포 1~2개의 개폐업만으로도 비율이 크게 튈 수 있음 — 실제 쇠퇴 신호가 아니라 표본이 작아서 생기는 통계적 노이즈일 위험이 있음.

**결정된 기준**: `전체_점포_수 ≥ 30`인 상권만 위험 판정 대상으로 인정. 미만인 상권은 위험상권 리스트에서 완전히 제외한다 (별도 표시 아님).

In [23]:
# df_위험상권(STEP 2에서 만든, 필터 적용 전 299개)에 최소 표본 기준 적용
최소_표본_임계값 = 30
표본_필터 = df_위험상권["전체_점포_수"] >= 최소_표본_임계값

df_위험상권_최종 = df_위험상권[표본_필터].copy()
df_위험상권_제외 = df_위험상권[~표본_필터]

print(f"필터 전 위험 상권: {len(df_위험상권)}개")
print(f"필터 후 위험 상권(최종): {len(df_위험상권_최종)}개")
print(f"표본 부족으로 제외된 상권: {len(df_위험상권_제외)}개")
print()
print("제외된 상권 목록 (전체_점포_수 오름차순):")
df_위험상권_제외.sort_values("전체_점포_수")[["상권_코드_명", "상권_구분_코드_명", "전체_점포_수", "KPI1_순증감율", "KPI2_폐업률증감"]]

필터 전 위험 상권: 299개
필터 후 위험 상권(최종): 285개
표본 부족으로 제외된 상권: 14개

제외된 상권 목록 (전체_점포_수 오름차순):


,상권_코드_명,상권_구분_코드_명,전체_점포_수,KPI1_순증감율,KPI2_폐업률증감
947,장안교,골목상권,11,-0.090909,0.007576
1567,연산군묘,골목상권,11,-0.272727,0.041958
1319,월곡역 4번,골목상권,13,-0.076923,0.005495
1799,수색성당,골목상권,17,-0.058824,0.003268
6383,강남시장(동작구 강남시장),전통시장,17,-0.117647,0.009804
1379,국립4.19민주묘지,골목상권,22,-0.045455,0.001976
1399,화계중학교,골목상권,22,-0.090909,0.003953
1127,신내IC북측,골목상권,24,-0.041667,0.003333
4295,강동성심병원,골목상권,27,-0.481481,0.037037
5891,백운시장,전통시장,27,-0.074074,0.002646


### 자치구 위험도 재계산 (필터 적용 후)

STEP 3에서 만든 `df_자치구_위험도`는 필터 적용 전(299개) 기준이었음. 최종 위험상권(`df_위험상권_최종`, 285개)으로 다시 계산해서 순위가 얼마나 바뀌는지 확인한다.

In [24]:
# STEP 3와 동일한 방식으로 자치구 위험도 재계산 (df_위험상권_최종 사용)
df_위험상권_구별_최종 = pd.merge(
    df_위험상권_최종[["상권_코드", "상권_코드_명"]],
    df_master[["상권_코드", "자치구_코드_명"]],
    on="상권_코드",
    how="left",
)

자치구별_위험개수_최종 = df_위험상권_구별_최종.groupby("자치구_코드_명").size().rename("위험_상권_수")
df_자치구_위험도_최종 = pd.concat([자치구별_전체상권수, 자치구별_위험개수_최종], axis=1).fillna(0)
df_자치구_위험도_최종["위험_상권_수"] = df_자치구_위험도_최종["위험_상권_수"].astype(int)
df_자치구_위험도_최종["위험_비율"] = df_자치구_위험도_최종["위험_상권_수"] / df_자치구_위험도_최종["전체_상권_수"]

print("=== 필터 적용 후 자치구 위험 비율 Top 10 (최종) ===")
print(df_자치구_위험도_최종.sort_values("위험_비율", ascending=False).head(10))
print()
print("참고: 필터 적용 전(STEP 3, 299개 기준) Top 10과 비교")
print(df_자치구_위험도.sort_values("위험_비율", ascending=False).head(10)[["전체_상권_수", "위험_상권_수", "위험_비율"]])

=== 필터 적용 후 자치구 위험 비율 Top 10 (최종) ===
          전체_상권_수  위험_상권_수     위험_비율
자치구_코드_명                            
동작구            57       15  0.263158
강동구            60       15  0.250000
양천구            56       13  0.232143
구로구            60       13  0.216667
송파구            70       15  0.214286
서대문구           62       13  0.209677
강서구            70       14  0.200000
은평구            68       13  0.191176
영등포구           96       18  0.187500
성동구            55       10  0.181818

참고: 필터 적용 전(STEP 3, 299개 기준) Top 10과 비교
          전체_상권_수  위험_상권_수     위험_비율
자치구_코드_명                            
동작구            57       17  0.298246
강동구            60       16  0.266667
도봉구            43       10  0.232558
양천구            56       13  0.232143
구로구            60       13  0.216667
송파구            70       15  0.214286
서대문구           62       13  0.209677
은평구            68       14  0.205882
강서구            70       14  0.200000
중랑구            61       12  0.196721
